#  Sales Performance Dashboard - Data Analytics Project

**Author:** Hoàng Yến
**Dataset:** Superstore Sales Dataset from Kaggle (9,994 rows × 21 columns)
**Objective:** Phân tích doanh thu, lợi nhuận, xây dựng insights và dashboard

---

##  Mục tiêu dự án
1. Phân tích doanh thu, lợi nhuận theo thời gian, khu vực, danh mục sản phẩm
2. Xác định top/bottom performing products và regions
3. Xây dựng dashboard tương tác cho việc theo dõi KPI
4. Đề xuất insights dựa trên dữ liệu


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Setup style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print('Libraries imported successfully!')

## 1. Data Collection

Dataset: [Superstore Dataset Final](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final) từ Kaggle
- **Rows:** 9,994
- **Columns:** 21
- **Fields:** Order info, Customer info, Product info, Financial metrics

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('data/superstore_raw.csv', encoding='utf-8-sig')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicated rows: {df.duplicated().sum()}")
df.head()

## 2️. Data Cleaning & Preparation

Giai đoạn tốn 60-80% thời gian của Data Analyst:
- Chuyển đổi kiểu dữ liệu ngày tháng
- Tạo cột Profit Margin
- Xử lý outliers bằng IQR method

In [ ]:
# Chuyển đổi kiểu dữ liệu ngày tháng
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Tạo cột Profit Margin
df['Profit_Margin'] = df['Profit'] / df['Sales']

# Xử lý outliers bằng IQR method
Q1 = df['Profit'].quantile(0.25)
Q3 = df['Profit'].quantile(0.75)
IQR = Q3 - Q1

df_clean = df[(df['Profit'] >= Q1 - 1.5*IQR) & (df['Profit'] <= Q3 + 1.5*IQR)].copy()

print(f"Dữ liệu gốc: {len(df)} dòng")
print(f"Dữ liệu sau cleaning: {len(df_clean)} dòng")
print(f"Đã loại bỏ: {len(df) - len(df_clean)} dòng ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

# Lưu dữ liệu đã làm sạch
df_clean.to_csv('data/superstore_cleaned.csv', index=False)
print("\n Saved cleaned dataset!")

## 3️. Data Analysis with SQL

Sử dụng SQLite để truy vấn và tổng hợp dữ liệu:

In [ ]:
# Tạo SQLite database
conn = sqlite3.connect(':memory:')
df_clean.to_sql('superstore', conn, index=False, if_exists='replace')

# Query 1: Doanh thu theo danh mục
query1 = """
SELECT Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit_Margin), 4) AS Avg_Margin
FROM superstore
GROUP BY Category
ORDER BY Total_Sales DESC;
"""

# Query 2: Top 5 states
query2 = """
SELECT State, ROUND(SUM(Profit), 2) AS Total_Profit
FROM superstore
GROUP BY State
ORDER BY Total_Profit DESC
LIMIT 5;
"""

# Query 3: Monthly trend
query3 = """
SELECT strftime('%Y-%m', "Order Date") AS Month,
    ROUND(SUM(Sales), 2) AS Monthly_Sales
FROM superstore
GROUP BY Month
ORDER BY Month;
"""

print("=== DOANH THU THEO DANH MỤC ===")
print(pd.read_sql(query1, conn))
print("\n=== TOP 5 BANG LỢI NHUẬN CAO NHẤT ===")
print(pd.read_sql(query2, conn))
print("\n=== XU HƯỚNG DOANH THU THEO THÁNG (10 dòng cuối) ===")
print(pd.read_sql(query3, conn).tail(10))

## 4️. Data Visualization

### 4.1 Doanh thu theo danh mục sản phẩm

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cat_data = pd.read_sql(
    "SELECT Category, SUM(Sales) as Total_Sales FROM superstore GROUP BY Category ORDER BY Total_Sales DESC", conn)
bars = ax.bar(cat_data['Category'], cat_data['Total_Sales'], color=['#2E86AB', '#A23B72', '#F18F01'])
ax.set_title('Doanh thu theo danh mục sản phẩm', fontsize=14, fontweight='bold')
ax.set_ylabel('Tổng doanh thu ($)')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'${height:,.0f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('images/sales_by_category.png', dpi=300)
plt.show()

### 4.2 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_clean[['Sales', 'Quantity', 'Discount', 'Profit', 'Profit_Margin']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, square=True, fmt='.2f')
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=300)
plt.show()

### 4.3 Xu hướng doanh thu theo thời gian

In [ ]:
monthly = pd.read_sql("""
    SELECT strftime('%Y-%m', "Order Date") AS Month,
           SUM(Sales) AS Sales, SUM(Profit) AS Profit
    FROM superstore GROUP BY Month ORDER BY Month
""", conn)
monthly['Month_dt'] = pd.to_datetime(monthly['Month'])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly['Month_dt'], monthly['Sales'], marker='o', label='Doanh thu', color='#2E86AB')
ax.plot(monthly['Month_dt'], monthly['Profit'], marker='s', label='Lợi nhuận', color='#F18F01')
ax.set_title('Xu hướng Doanh thu & Lợi nhuận', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('images/sales_trend.png', dpi=300)
plt.show()

### 4.4 Regional Performance

In [ ]:
regional = pd.read_sql("""
    SELECT Region, SUM(Sales) AS Sales, SUM(Profit) AS Profit
    FROM superstore GROUP BY Region ORDER BY Sales DESC
""", conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
ax1.bar(regional['Region'], regional['Sales'], color=colors)
ax1.set_title('Doanh thu theo Region')
ax2.bar(regional['Region'], regional['Profit'], color=colors)
ax2.set_title('Lợi nhuận theo Region')
plt.suptitle('Regional Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/regional_performance.png', dpi=300)
plt.show()

### 4.5 Discount Impact Analysis

In [ ]:
discount_data = pd.read_sql("""
    SELECT CASE 
        WHEN Discount = 0 THEN '0%'
        WHEN Discount <= 0.2 THEN '1-20%'
        WHEN Discount <= 0.4 THEN '21-40%'
        WHEN Discount <= 0.6 THEN '41-60%'
        ELSE '61-80%'
    END AS Discount_Level,
    AVG(Sales) AS Avg_Sales, AVG(Profit) AS Avg_Profit, AVG(Profit_Margin) AS Avg_Margin
    FROM superstore GROUP BY Discount_Level ORDER BY MIN(Discount)
""", conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.bar(discount_data['Discount_Level'], discount_data['Avg_Profit'], color=['#2E86AB', '#6A994E', '#F18F01', '#C73E1D', '#6C757D'])
ax1.set_title('Lợi nhuận TB theo mức giảm giá')
ax2.bar(discount_data['Discount_Level'], discount_data['Avg_Margin'], color=['#2E86AB', '#6A994E', '#F18F01', '#C73E1D', '#6C757D'])
ax2.set_title('Profit Margin TB theo mức giảm giá')
plt.suptitle('Discount Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/discount_impact.png', dpi=300)
plt.show()

### 4.6 Customer Segment Analysis

In [ ]:
segment = pd.read_sql("""
    SELECT Segment, SUM(Sales) AS Sales FROM superstore GROUP BY Segment
""", conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors_seg = ['#2E86AB', '#A23B72', '#F18F01']
ax1.pie(segment['Sales'], labels=segment['Segment'], autopct='%1.1f%%', colors=colors_seg)
ax1.set_title('Tỷ trọng Doanh thu')

seg_margin = pd.read_sql("""
    SELECT Segment, AVG(Profit_Margin) AS Avg_Margin FROM superstore GROUP BY Segment
""", conn)
bars = ax2.bar(seg_margin['Segment'], seg_margin['Avg_Margin'], color=colors_seg)
ax2.set_title('Profit Margin theo Segment')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{bar.get_height():.2%}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('images/segment_analysis.png', dpi=300)
plt.show()

### 4.7 Top 10 States by Profit

In [ ]:
top10 = pd.read_sql("""
    SELECT State, SUM(Profit) AS Total_Profit
    FROM superstore GROUP BY State ORDER BY Total_Profit DESC LIMIT 10
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Spectral(np.linspace(0.1, 0.9, 10))
ax.barh(top10['State'][::-1], top10['Total_Profit'][::-1], color=colors)
ax.set_title('Top 10 Bang có lợi nhuận cao nhất', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/top_states_profit.png', dpi=300)
plt.show()

## 5️. Key Findings & Insights

###  Kết quả phân tích chính:

1. **Office Supplies dẫn đầu doanh thu** ($283K, 36.8%) nhưng Profit Margin (16.6%) vượt trội so với Furniture (10.6%)
   - Furniture có doanh thu cao ($260K) nhưng lợi nhuận thấp nhất
   - Technology có Profit Margin tốt (14.6%) nhưng doanh thu thấp nhất

2. **California, New York, Washington** là 3 bang đóng góp lợi nhuận lớn nhất
   - California một mình chiếm $26.4K lợi nhuận (gần gấp đôi New York)

3. **Discount > 20% = Profit âm** 
   - Không có discount: Profit Margin 34.3%
   - Discount 1-20%: Profit Margin giảm còn 18.1%
   - Discount 21-40%: Profit Margin âm -10.9%
   - **Insight:** Chính sách giảm giá mạnh đang 'giết' lợi nhuận

4. **Consumer Segment chiếm 53.2% doanh thu** nhưng Home Office có Profit Margin cao nhất (16.8%)
   - Corporate: 15.4%, Consumer: 14.5%

5. **West Region** dẫn đầu cả doanh thu ($295K) và lợi nhuận ($38.6K)
   - South Region có doanh thu thấp nhất ($111K) nhưng lợi nhuận tương đương Central

###  Recommendations:
- **Tối ưu chính sách giảm giá:** Cân nhắc giảm discount rate xuống dưới 20%, đặc biệt cho Furniture
- **Tập trung vào Office Supplies:** Mở rộng danh mục này vì có profit margin tốt nhất
- **Khai thác Home Office segment:** Tăng cường marketing cho segment có margin cao nhất
- **Cải thiện Central Region:** Điều tra nguyên nhân lợi nhuận thấp tại khu vực này
